# Week 11, part 2 prep: getting two NYC datasets ready to layer

This notebook prepares the files for `week11_part2_scratch.ipynb`, so that notebook can be all about the map. It reads the raw files and saves four clean ones:

* `zip_population.csv`: the population and center point of each ZIP area
* `zip_areas.geojson`: the shape of each ZIP area
* `cd_housing_units.csv`: the number of housing units in each community district
* `nyc_districts.geojson`: the shape of each community district

**Where the raw files come from**

* **ZIP areas:** `data-by-modzcta.csv` and `MODZCTA_2010_WGS1984.geo.json`, from the NYC Health Department's [coronavirus-data repository](https://github.com/nychealth/coronavirus-data) on GitHub. We only want the population column and the shapes. "MODZCTA" means *modified ZIP code tabulation area*: the Census Bureau's version of ZIP codes, with a few tiny areas merged into their neighbors by NYC. They line up with ZIP codes, but not one-to-one. A few areas cover more than one ZIP code.
* **Community districts:** `Housing_Database_by_Community_District_20261102.csv` and `nyc_districts.geojson`, from the Week 10 folder.

In [ ]:
import json

import pandas as pd

## 1. ZIP area populations

The raw file has many columns about COVID-19 cases, which we don't need. We keep the ones we want, and give them clearer names.

In [ ]:
raw_zip = pd.read_csv("data-by-modzcta.csv")

print(raw_zip.shape)
raw_zip.head(3)

`POP_DENOMINATOR` is the population estimate the Health Department uses to calculate rates. It has decimals, because it's an estimate, so we round it to whole people. `lat` and `lon` are a center point for each area, which we'll use to place a bubble.

The ZIP code is a **text** value, not a number (some start with a zero in other parts of the country, and we never add them up), so we save it as text.

In [ ]:
zips = raw_zip.rename(columns={
    "MODIFIED_ZCTA": "zip",
    "NEIGHBORHOOD_NAME": "neighborhood",
    "BOROUGH_GROUP": "borough",
    "label": "zip_codes",           # like "10001, 10118" when an area has more than one ZIP code
    "POP_DENOMINATOR": "population",
})

zips = zips[["zip", "zip_codes", "neighborhood", "borough", "lat", "lon", "population"]].copy()

zips["zip"] = zips["zip"].astype(str)
zips["population"] = zips["population"].round().astype(int)

print(len(zips), "areas,", zips["population"].sum(), "people")
zips.sort_values("population").head()

In [ ]:
zips.to_csv("zip_population.csv", index=False)

## 2. ZIP area shapes

The shapes file is GeoJSON, which we read with `json.load()`. Each shape's `properties` has a `MODZCTA` code. One shape, `99999`, is a placeholder for everything outside the city, and it has no population, so we drop it. Then we keep only the ZIP code in each shape's properties, and call it `zip`, to match the table.

In [ ]:
with open("MODZCTA_2010_WGS1984.geo.json", "r", encoding="utf-8") as file:
    zip_shapes = json.load(file)

print(len(zip_shapes["features"]), "shapes")
print(zip_shapes["features"][0]["properties"])

In [ ]:
# keep a shape only if it isn't the placeholder: build a new list, and add each shape that passes the test
kept = []

for feature in zip_shapes["features"]:
    if feature["properties"]["MODZCTA"] != "99999":
        kept.append(feature)

zip_shapes["features"] = kept

# replace each shape's properties with just the ZIP code, under the name "zip"
for feature in zip_shapes["features"]:
    feature["properties"] = {"zip": feature["properties"]["MODZCTA"]}

print(len(zip_shapes["features"]), "shapes")

# check: every shape should match a row in the table, and the other way around.
# First, collect the ZIP code of every shape in a set (a collection with no repeats)
shape_zips = set()

for feature in zip_shapes["features"]:
    shape_zips.add(feature["properties"]["zip"])

# then compare it with the set of ZIP codes in the table
print("shapes and table match:", shape_zips == set(zips["zip"]))

In [ ]:
with open("zip_areas.geojson", "w", encoding="utf-8") as file:
    json.dump(zip_shapes, file)

## 3. Housing units by community district

We use the same housing file as in Week 10. The **total number of housing units** is `cenunits20`, the count from the 2020 Census. Since then, new buildings have been completed (`comp2021` through `comp2026`), so we also save a second, more recent estimate: the 2020 count plus the net units completed since.

As in Week 10, the code is `commntydst`: the first digit is the borough, and the last two digits are the district number. Numbers above 18 are parks and airports, not districts.

In [ ]:
housing = pd.read_csv("../week10/Housing_Database_by_Community_District_20261102.csv", thousands=",")

boroughs = {1: "Manhattan", 2: "Bronx", 3: "Brooklyn", 4: "Queens", 5: "Staten Island"}

housing["borough"] = (housing["commntydst"] // 100).map(boroughs)
housing["number"] = housing["commntydst"] % 100
housing["label"] = housing["borough"] + " CD " + housing["number"].astype(str)
housing["is_district"] = housing["number"] <= 18

# the net units completed since the 2020 census
years_since = []

for year in range(2021, 2027):          # range(2021, 2027) stops before 2027
    years_since.append(f"comp{year}")   # "comp2021", "comp2022", ...
housing["completed_since_2020"] = housing[years_since].sum(axis=1)

housing["housing_units"] = housing["cenunits20"]
housing["units_now"] = housing["cenunits20"] + housing["completed_since_2020"]

units = housing[["commntydst", "label", "borough", "is_district", "housing_units", "completed_since_2020", "units_now"]]

print(units["housing_units"].sum(), "housing units in 2020")
units[units["is_district"]].sort_values("housing_units", ascending=False).head()

In [ ]:
units.to_csv("cd_housing_units.csv", index=False)

## 4. Community district shapes

The Week 10 shapes file also has the longitude and latitude of each center, which we don't need here, so we keep only the district code.

In [ ]:
with open("../week10/nyc_districts.geojson", "r", encoding="utf-8") as file:
    district_shapes = json.load(file)

for feature in district_shapes["features"]:
    feature["properties"] = {"commntydst": feature["properties"]["commntydst"]}

print(len(district_shapes["features"]), "shapes")

with open("nyc_districts.geojson", "w", encoding="utf-8") as file:
    json.dump(district_shapes, file)

Now `week11_part2_scratch.ipynb` can load these four files.